## RAG Fusion Pipeline

This notebook demonstrates a complete RAG (Retrieval-Augmented Generation) pipeline built on top of our custom `RAGFusion` class.

The pipeline uses LLM-generated sub-queries to improve retrieval quality, then fuses the results using Reciprocal Rank Fusion (RRF).

### Steps covered

1. Load the source PDF
2. Split documents into chunks
3. Generate embeddings and store in ChromaDB
4. Create a similarity-search retriever
5. Apply RAG Fusion (sub-query generation + RRF)
6. Augmentation — build context from retrieved documents
7. Generation — produce a grounded answer using an LLM

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from rag_fusion import RAGFusion

/tmp/ipykernel_601592/2620883329.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [5]:
llm = ChatOllama(
    model="smollm2:360m",
    temperature=0

) 
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)


In [7]:
loader = PyPDFLoader("notebooklm_rag.pdf")
pages = loader.load()

print(f"Loaded {len(pages)} page(s) from the PDF.")

Loaded 3 page(s) from the PDF.


In [13]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(pages)

print(f"Split into {len(chunks)} chunk(s).")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="notebooklm_rag"
)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

Split into 19 chunk(s).


In [14]:
# Build the RAG Fusion pipeline: LLM generates 2 sub-queries, retrieves docs for each, then fuses
rag_fusion = RAGFusion.from_llm(
    llm=llm,
    retriever=retriever,
    num_subqueries=2,
    k=3
)

In [15]:
query = "How does NotebookLM retrieve relevant information from uploaded documents?"

# This generates sub-queries, retrieves docs for each, and returns RRF-ranked results
fused_docs = rag_fusion.invoke(query)

print(f"Retrieved {len(fused_docs)} fused document(s).")
for i, doc in enumerate(fused_docs):
    print(f"\n--- Document {i + 1} ---")
    print(doc.page_content)

Retrieved 3 fused document(s).

--- Document 1 ---
When a user uploads a document to NotebookLM, the system begins an automatic indexing process. The
document is first parsed to extract its raw text content. For PDFs, this involves optical character recognition
(OCR) if the document contains scanned pages, or direct text extraction for digital PDFs. The extracted text is
then cleaned and normalized to remove formatting artifacts.
Next, the text is split into overlapping chunks using a strategy that preserves semantic coherence. Rather

--- Document 2 ---
material provided by the user, making it particularly useful for researchers, students, and knowledge workers
who need to work deeply with specific documents.
The tool was first introduced in 2023 and has since gained significant attention for its ability to synthesize
information from multiple sources simultaneously. Users can upload PDFs, Google Docs, YouTube
transcripts, and other formats, and NotebookLM will treat these as the auth

In [16]:
context = "\n\n".join([doc.page_content for doc in fused_docs])

print(context)

When a user uploads a document to NotebookLM, the system begins an automatic indexing process. The
document is first parsed to extract its raw text content. For PDFs, this involves optical character recognition
(OCR) if the document contains scanned pages, or direct text extraction for digital PDFs. The extracted text is
then cleaned and normalized to remove formatting artifacts.
Next, the text is split into overlapping chunks using a strategy that preserves semantic coherence. Rather

material provided by the user, making it particularly useful for researchers, students, and knowledge workers
who need to work deeply with specific documents.
The tool was first introduced in 2023 and has since gained significant attention for its ability to synthesize
information from multiple sources simultaneously. Users can upload PDFs, Google Docs, YouTube
transcripts, and other formats, and NotebookLM will treat these as the authoritative knowledge base for
answering questions.

model can reference

In [17]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Use ONLY the context provided below to answer the question.
Be clear, concise, and accurate in your response.
If the answer is not present in the context, say "I don't know" - do not make up an answer.

Context:
{context}

Question: {question}

Answer:
""")

# Chain: prompt -> LLM
generation_chain = prompt | llm

response = generation_chain.invoke({"context": context, "question": query})

print(response.content)

NotebookLM processes documents by first parsing the raw text content to extract the relevant information. This involves OCR for PDFs and direct text extraction for digital PDFs. The extracted text is then cleaned and normalized to remove formatting artifacts. The chunks are split into overlapping sections to preserve semantic coherence. The tool was first introduced in 2023 and has since gained significant attention for its ability to synthesize information from multiple sources simultaneously. Users can upload PDFs, Google Docs, YouTube transcripts, and other formats, and NotebookLM will treat these as the authoritative knowledge base for answering questions.
